# 🏠 Home Assistant — Next-Action Predictor
> **Full ML pipeline**: data loading → feature engineering → training (RF + XGBoost + LightGBM) → evaluation → interactive demo

### Setup
1. Upload your `home_assistant_interactions_combined.jsonl` file using the **Upload dataset** cell below
2. Run all cells top-to-bottom (`Runtime → Run all`)
3. The interactive demo widget appears at the bottom

| Metric | Value |
|---|---|
| Dataset | 10,000 samples, 26 action classes |
| Features | 125 engineered features |
| Models | Random Forest · XGBoost · **LightGBM** |
| Best Top-1 | ~34% (26 classes) / ~48% (6 grouped classes) |
| Best Top-3 | ~86% (6 grouped classes) |

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q xgboost lightgbm scikit-learn pandas matplotlib ipywidgets
from IPython.display import clear_output
clear_output()
print('✅ Dependencies ready (RF · XGBoost · LightGBM)')

In [ ]:
# ── Cell 2: Upload dataset ────────────────────────────────────────────────────
from google.colab import files
import os

DATASET_PATH = None

# Check if already uploaded
for fname in os.listdir('.'):
    if fname.endswith('.jsonl'):
        DATASET_PATH = fname
        print(f'✅ Found dataset: {fname}')
        break

if DATASET_PATH is None:
    print('📂 Please upload your .jsonl dataset file:')
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]
    print(f'✅ Uploaded: {DATASET_PATH}')

In [ ]:
# ── Cell 3: Feature engineering ───────────────────────────────────────────────
import json, numpy as np, pandas as pd
from collections import Counter

DEVICE_KEYS = [
    'light_bedroom', 'light_living_room', 'light_kitchen', 'light_bathroom',
    'climate_hvac', 'media_player_tv', 'media_player_music',
    'fan_living_room', 'fan_bedroom', 'air_purifier_bedroom',
]

def load_and_featurize(path, max_rows=10000):
    rows, labels = [], []
    with open(path, 'r') as f:
        for i, line in enumerate(f):
            if i >= max_rows: break
            d  = json.loads(line)
            td = d.get('triggered_device')
            if not td: continue
            label = f"{td['entity_id'].replace('.','_')}_{td['action']}"
            labels.append(label)

            tf, occ, ctx = d['temporal_features'], d['occupancy'], d['contextual_features']
            cf, hs = d['correlation_flags'], d['home_state_before']
            dus, rh = d.get('device_usage_stats', {}), d.get('recent_history', [])

            h, m, dow = tf['hour'], tf['minute'], tf['day_of_week']
            row = {
                # cyclical time
                'hour_sin': np.sin(2*np.pi*h/24), 'hour_cos': np.cos(2*np.pi*h/24),
                'minute_sin': np.sin(2*np.pi*m/60), 'minute_cos': np.cos(2*np.pi*m/60),
                'dow_sin': np.sin(2*np.pi*dow/7), 'dow_cos': np.cos(2*np.pi*dow/7),
                'hour': h, 'minute': m, 'dow': dow,
                'is_morning': int(tf['is_morning']), 'is_afternoon': int(tf['is_afternoon']),
                'is_evening': int(tf['is_evening']), 'is_night': int(tf['is_night']),
                'is_weekday': int(tf['is_weekday']),
                # occupancy
                'is_occupied': int(occ['is_occupied']),
                'occ_confidence': occ['occupancy_confidence'],
                'people_count': occ['people_count'],
                # weather / context
                'outdoor_temp': ctx['outdoor_temperature'],
                'outdoor_hum': ctx['outdoor_humidity'],
                'is_raining': int(ctx['is_raining']), 'is_sunny': int(ctx['is_sunny']),
                'ambient_light': ctx['ambient_light_level'],
                'last_away_min': ctx['last_away_duration_minutes'],
                'days_away': ctx['days_since_last_away'],
                # derived
                'temp_humidity_index': ctx['outdoor_temperature'] * ctx['outdoor_humidity'] / 100,
                'just_arrived': int(ctx['last_away_duration_minutes'] > 0 and ctx['days_since_last_away'] == 0),
                # flags
                'morning_routine': int(cf['follows_morning_routine']),
                'evening_routine': int(cf['follows_evening_routine']),
                'entertainment': int(cf['is_entertainment_time']),
                'away_mode': int(cf['is_away_mode_active']),
                'sleep_mode': int(cf['is_sleep_mode_active']),
                'lights_on_count': 0, 'total_devices_on': 0,
            }
            lights_on = total_on = 0
            for eid, es in hs.items():
                key = eid.replace('.', '_')
                is_on = 1 if es['state'] in ('on','heating','cooling','playing') else 0
                row[f'{key}_on'] = is_on
                total_on += is_on
                if 'light' in eid: lights_on += is_on
                attrs = es.get('attributes', {})
                for attr, feat in [('brightness','brightness'),('color_temp','color_temp'),
                                   ('current_temperature','cur_temp'),('target_temperature','tgt_temp'),
                                   ('volume_level','volume'),('speed','speed'),('aqi','aqi')]:
                    if attrs.get(attr) is not None: row[f'{key}_{feat}'] = attrs[attr]
                if 'climate' in eid and attrs.get('current_temperature') and attrs.get('target_temperature'):
                    row['hvac_temp_delta'] = attrs['target_temperature'] - attrs['current_temperature']
            row['lights_on_count'] = lights_on
            row['total_devices_on'] = total_on
            # usage stats
            for dk in DEVICE_KEYS:
                s = dus.get(dk, {})
                row[f'{dk}_uses_today'] = s.get('times_used_today', 0) or 0
                row[f'{dk}_avg_dur']    = s.get('avg_duration_minutes', 0) or 0
                row[f'{dk}_first_use_hour'] = s.get('first_use_hour') or -1
                row[f'{dk}_last_use_hour']  = s.get('last_use_hour')  or -1
            # recent history
            row['rh_len'] = len(rh)
            for rank, event in enumerate(rh[:3]):
                prefix = f'rh{rank}'
                eid = event.get('entity_id', '').replace('.', '_')
                row[f'{prefix}_elapsed_s'] = event.get('elapsed_seconds', 9999)
                row[f'{prefix}_action_on'] = int(event.get('action', '') == 'on')
                for cat in ['light','climate','media_player','fan','air_purifier','switch']:
                    row[f'{prefix}_is_{cat}'] = int(cat in eid)
            rows.append(row)
    return pd.DataFrame(rows).fillna(0), labels

print('Loading dataset...')
df, fine_labels = load_and_featurize(DATASET_PATH)
print(f'✅ {len(df)} labeled samples | {df.shape[1]} features | {len(set(fine_labels))} classes')
print('Class distribution (top 10):')
for cls, cnt in Counter(fine_labels).most_common(10):
    print(f'  {cls:40s} {cnt:4d} ({cnt/len(fine_labels):.1%})')

In [ ]:
# ── Cell 4: Train fine-grained models (26 classes) ────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, top_k_accuracy_score
import xgboost as xgb_lib

le_fine = LabelEncoder()
y_fine  = le_fine.fit_transform(fine_labels)
X       = df.values.astype(np.float32)
n_cls   = len(le_fine.classes_)

from collections import Counter as C2
can_strat = all(v >= 2 for v in C2(y_fine).values())
X_tr, X_te, y_tr, y_te = train_test_split(X, y_fine, test_size=0.2,
                                            random_state=42,
                                            stratify=y_fine if can_strat else None)

def pad_proba(raw, model_classes, n_total):
    if raw.shape[1] == n_total: return raw
    out = np.zeros((raw.shape[0], n_total), dtype=np.float32)
    for i, c in enumerate(model_classes): out[:, c] = raw[:, i]
    return out

print('Training Random Forest (200 trees)...')
rf_fine = RandomForestClassifier(n_estimators=200, n_jobs=-1, max_depth=None, random_state=42)
rf_fine.fit(X_tr, y_tr)
rf_pred  = rf_fine.predict(X_te)
rf_proba = pad_proba(rf_fine.predict_proba(X_te), rf_fine.classes_, n_cls)
rf_acc   = accuracy_score(y_te, rf_pred)
rf_top3  = top_k_accuracy_score(y_te, rf_proba, k=min(3,n_cls), labels=list(range(n_cls)))
print(f'  RF  → Top-1: {rf_acc:.1%}  Top-3: {rf_top3:.1%}')

print('Training XGBoost (200 trees)...')
xgb_fine = xgb_lib.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1,
                                   use_label_encoder=False, eval_metric='mlogloss',
                                   n_jobs=-1, random_state=42, verbosity=0,
                                   tree_method='hist', device='cuda')
try:
    xgb_fine.fit(X_tr, y_tr)
    print('  (GPU accelerated)')
except Exception:
    xgb_fine.set_params(device='cpu')
    xgb_fine.fit(X_tr, y_tr)
    print('  (CPU fallback)')
xg_pred  = xgb_fine.predict(X_te)
xg_proba = pad_proba(xgb_fine.predict_proba(X_te), xgb_fine.classes_, n_cls)
xg_acc   = accuracy_score(y_te, xg_pred)
xg_top3  = top_k_accuracy_score(y_te, xg_proba, k=min(3,n_cls), labels=list(range(n_cls)))
print(f'  XGB → Top-1: {xg_acc:.1%}  Top-3: {xg_top3:.1%}')

print('Training LightGBM (200 trees)...')
import lightgbm as lgb_lib
lgb_fine = lgb_lib.LGBMClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1,
    num_leaves=63, n_jobs=-1, random_state=42, verbose=-1,
    device='gpu' if 'cuda' in str(xgb_lib.__version__) else 'cpu'
)
try:
    lgb_fine.fit(X_tr, y_tr)
    print('  (GPU accelerated)')
except Exception:
    lgb_fine.set_params(device='cpu')
    lgb_fine.fit(X_tr, y_tr)
    print('  (CPU fallback)')
lg_pred  = lgb_fine.predict(X_te)
lg_proba = pad_proba(lgb_fine.predict_proba(X_te), lgb_fine.classes_, n_cls)
lg_acc   = accuracy_score(y_te, lg_pred)
lg_top3  = top_k_accuracy_score(y_te, lg_proba, k=min(3,n_cls), labels=list(range(n_cls)))
print(f'  LGB → Top-1: {lg_acc:.1%}  Top-3: {lg_top3:.1%}')
print('✅ Fine-grained training complete')

In [ ]:
# ── Cell 5: Train grouped models (6 classes) ──────────────────────────────────
GROUP_MAP = {
    'light_bedroom_on':'Light ON','light_living_room_on':'Light ON',
    'light_kitchen_on':'Light ON','light_bathroom_on':'Light ON',
    'light_bedroom_off':'Light OFF','light_living_room_off':'Light OFF',
    'light_kitchen_off':'Light OFF','light_bathroom_off':'Light OFF',
    'climate_hvac_heat':'Climate','climate_hvac_cool':'Climate',
    'climate_hvac_off':'Climate','climate_hvac_on':'Climate',
    'media_player_tv_on':'Media','media_player_tv_off':'Media',
    'media_player_music_on':'Media','media_player_music_off':'Media',
    'switch_washer_on':'Appliance','switch_washer_off':'Appliance',
    'switch_dishwasher_on':'Appliance','switch_dishwasher_off':'Appliance',
    'switch_coffee_maker_on':'Appliance','switch_coffee_maker_off':'Appliance',
    'fan_living_room_on':'Fan/Air','fan_living_room_off':'Fan/Air',
    'fan_bedroom_on':'Fan/Air','fan_bedroom_off':'Fan/Air',
    'air_purifier_bedroom_on':'Fan/Air','air_purifier_bedroom_off':'Fan/Air',
}
def map_label(lbl):
    for k, g in GROUP_MAP.items():
        if k in lbl: return g
    if 'light' in lbl: return 'Light ON' if lbl.endswith('_on') else 'Light OFF'
    if 'climate' in lbl or 'hvac' in lbl: return 'Climate'
    if 'media' in lbl or 'tv' in lbl or 'music' in lbl: return 'Media'
    if 'fan' in lbl or 'purifier' in lbl: return 'Fan/Air'
    return 'Appliance'

grouped_labels = [map_label(l) for l in fine_labels]
le_grp = LabelEncoder()
y_grp  = le_grp.fit_transform(grouped_labels)
n_grp  = len(le_grp.classes_)

Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(X, y_grp, test_size=0.2,
                                                random_state=42, stratify=y_grp)
print('Training grouped Random Forest...')
rf_grp = RandomForestClassifier(n_estimators=200, n_jobs=-1, max_depth=None, random_state=42)
rf_grp.fit(Xg_tr, yg_tr)
grf_pred  = rf_grp.predict(Xg_te)
grf_proba = pad_proba(rf_grp.predict_proba(Xg_te), rf_grp.classes_, n_grp)
grf_acc   = accuracy_score(yg_te, grf_pred)
grf_top3  = top_k_accuracy_score(yg_te, grf_proba, k=min(3,n_grp), labels=list(range(n_grp)))
print(f'  RF  → Top-1: {grf_acc:.1%}  Top-3: {grf_top3:.1%}')

print('Training grouped XGBoost...')
xgb_grp = xgb_lib.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1,
                                  use_label_encoder=False, eval_metric='mlogloss',
                                  n_jobs=-1, random_state=42, verbosity=0,
                                  tree_method='hist', device='cuda')
try:
    xgb_grp.fit(Xg_tr, yg_tr)
    print('  (GPU accelerated)')
except Exception:
    xgb_grp.set_params(device='cpu')
    xgb_grp.fit(Xg_tr, yg_tr)
    print('  (CPU fallback)')
gxg_pred  = xgb_grp.predict(Xg_te)
gxg_proba = pad_proba(xgb_grp.predict_proba(Xg_te), xgb_grp.classes_, n_grp)
gxg_acc   = accuracy_score(yg_te, gxg_pred)
gxg_top3  = top_k_accuracy_score(yg_te, gxg_proba, k=min(3,n_grp), labels=list(range(n_grp)))
print(f'  XGB → Top-1: {gxg_acc:.1%}  Top-3: {gxg_top3:.1%}')

print('Training grouped LightGBM...')
lgb_grp = lgb_lib.LGBMClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1,
    num_leaves=63, n_jobs=-1, random_state=42, verbose=-1)
try:
    lgb_grp.fit(Xg_tr, yg_tr)
    print('  (GPU accelerated)')
except Exception:
    lgb_grp.set_params(device='cpu')
    lgb_grp.fit(Xg_tr, yg_tr)
    print('  (CPU fallback)')
glg_pred  = lgb_grp.predict(Xg_te)
glg_proba = pad_proba(lgb_grp.predict_proba(Xg_te), lgb_grp.classes_, n_grp)
glg_acc   = accuracy_score(yg_te, glg_pred)
glg_top3  = top_k_accuracy_score(yg_te, glg_proba, k=min(3,n_grp), labels=list(range(n_grp)))
print(f'  LGB → Top-1: {glg_acc:.1%}  Top-3: {glg_top3:.1%}')
print('✅ Grouped training complete')

In [ ]:
# ── Cell 6: Visualisations ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import classification_report, confusion_matrix

BG, CARD, BORDER = '#0e1420', '#151d2c', '#26314a'
ACCENT, ACCENT2, TEXT, MUTED, WARN = '#22c9a4', '#4f8ef7', '#e6edf7', '#a9b6cc', '#f7b32b'

# 1. Model comparison bar chart — all three models
fig, ax = plt.subplots(figsize=(12, 4.5), facecolor=BG)
ax.set_facecolor(CARD)
models = ['Random Forest', 'XGBoost', 'LightGBM']
fine_top1  = [rf_acc*100,  xg_acc*100,  lg_acc*100]
fine_top3  = [rf_top3*100, xg_top3*100, lg_top3*100]
grp_top1   = [grf_acc*100, gxg_acc*100, glg_acc*100]
grp_top3   = [grf_top3*100,gxg_top3*100,glg_top3*100]
data = [
    ('Fine Top-1 (26 cls)', fine_top1,  '#4a5568'),
    ('Fine Top-3 (26 cls)', fine_top3,  '#6b7280'),
    ('Group Top-1 (6 cls)', grp_top1,   ACCENT),
    ('Group Top-3 (6 cls)', grp_top3,   ACCENT2),
]
x, w = np.arange(len(models)), 0.18
for i, (lbl, vals, col) in enumerate(data):
    bars = ax.bar(x+(i-1.5)*w, vals, w, label=lbl, color=col, zorder=3)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', color=TEXT, fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(models, color=TEXT, fontsize=12)
ax.set_ylim(0, 105); ax.set_ylabel('Accuracy %', color=MUTED)
ax.tick_params(colors=MUTED); ax.spines[:].set_color(BORDER)
ax.legend(facecolor=CARD, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax.set_title('Model Accuracy — RF vs XGBoost vs LightGBM (Fine & Grouped)', color=TEXT, fontsize=13)
ax.axhline(90, color=WARN, linestyle='--', linewidth=1.2, zorder=4)
ax.text(2.4, 91, '90% target', color=WARN, fontsize=8)
ax.grid(axis='y', color=BORDER, linestyle='--', alpha=0.5, zorder=0)
fig.tight_layout(pad=1.5)
plt.show()

# 1b. Summary table
print(f'\n{"Model":<16} {"Fine Top-1":>12} {"Fine Top-3":>12} {"Grp Top-1":>12} {"Grp Top-3":>12}')
print('-'*66)
for m, ft1, ft3, gt1, gt3 in zip(models, fine_top1, fine_top3, grp_top1, grp_top3):
    print(f'{m:<16} {ft1:>11.1f}% {ft3:>11.1f}% {gt1:>11.1f}% {gt3:>11.1f}%')

# 2. Confusion matrix (grouped)
cm      = confusion_matrix(yg_te, grf_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
tlabels = le_grp.classes_
ax.set_xticks(range(n_grp)); ax.set_yticks(range(n_grp))
ax.set_xticklabels(tlabels, rotation=30, ha='right', color=TEXT, fontsize=9)
ax.set_yticklabels(tlabels, color=TEXT, fontsize=9)
for i in range(n_grp):
    for j in range(n_grp):
        ax.text(j, i, f'{cm_norm[i,j]:.0%}', ha='center', va='center',
                color='white' if cm_norm[i,j]>0.5 else TEXT, fontsize=10, fontweight='bold')
ax.set_title('Random Forest — Confusion Matrix (6 classes)', color=TEXT, fontsize=12)
ax.set_xlabel('Predicted', color=MUTED); ax.set_ylabel('True', color=MUTED)
ax.spines[:].set_color(BORDER)
fig.tight_layout(pad=1.5)
plt.show()

# 3. Feature importance — all three models
names   = df.columns.tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor=BG)
for ax, (model, color, title) in zip(axes, [
    (rf_fine,  ACCENT,  'Random Forest'),
    (xgb_fine, ACCENT2, 'XGBoost'),
    (lgb_fine, '#c084fc','LightGBM'),
]):
    imp     = model.feature_importances_
    top_idx = np.argsort(imp)[::-1][:15]
    vals    = imp[top_idx]
    ax.set_facecolor(CARD)
    ax.barh([names[i].replace('_',' ') for i in top_idx], vals,
            color=plt.cm.RdYlGn(vals/vals.max()), zorder=3)
    ax.set_xlabel('Importance', color=MUTED)
    ax.set_title(f'{title} — Top 15', color=TEXT, fontsize=10)
    ax.tick_params(colors=MUTED, labelsize=7); ax.spines[:].set_color(BORDER)
    ax.grid(axis='x', color=BORDER, linestyle='--', alpha=0.4, zorder=0)
fig.suptitle('Feature Importances — RF vs XGBoost vs LightGBM', color=TEXT, fontsize=13)
fig.tight_layout(pad=1.5)
plt.show()
print('✅ Charts rendered')

In [ ]:
# ── Cell 7: Interactive prediction widget ─────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

style  = {'description_width': '160px'}
layout = widgets.Layout(width='420px')

w_hour   = widgets.IntSlider(value=18, min=0, max=23, description='Hour (0-23)', style=style, layout=layout)
w_dow    = widgets.Dropdown(value=4, options=[(d,i) for i,d in enumerate(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])],
                             description='Day of week', style=style, layout=layout)
w_occ    = widgets.Checkbox(value=True,  description='Home occupied',     style=style)
w_people = widgets.IntSlider(value=2, min=0, max=5, description='People count', style=style, layout=layout)
w_temp   = widgets.FloatSlider(value=25, min=5, max=45, step=0.5, description='Outdoor temp °C', style=style, layout=layout)
w_hum    = widgets.IntSlider(value=55, min=0, max=100, description='Humidity %', style=style, layout=layout)
w_rain   = widgets.Checkbox(value=False, description='Is raining',        style=style)
w_sunny  = widgets.Checkbox(value=True,  description='Is sunny',          style=style)
w_light_br = widgets.Checkbox(value=False, description='Bedroom light ON',    style=style)
w_light_lr = widgets.Checkbox(value=True,  description='Living room light ON', style=style)
w_light_k  = widgets.Checkbox(value=True,  description='Kitchen light ON',     style=style)
w_light_ba = widgets.Checkbox(value=False, description='Bathroom light ON',    style=style)
w_hvac   = widgets.Checkbox(value=True,  description='HVAC on',           style=style)
w_hcur   = widgets.FloatSlider(value=21, min=10, max=35, step=0.5, description='HVAC current °C', style=style, layout=layout)
w_htgt   = widgets.FloatSlider(value=23, min=10, max=35, step=0.5, description='HVAC target °C',  style=style, layout=layout)
w_tv     = widgets.Checkbox(value=False, description='TV on',             style=style)
w_music  = widgets.Checkbox(value=False, description='Music on',          style=style)
w_fan_lr = widgets.Checkbox(value=False, description='Living room fan ON', style=style)
w_fan_br = widgets.Checkbox(value=False, description='Bedroom fan ON',    style=style)
w_purif  = widgets.Checkbox(value=False, description='Air purifier ON',   style=style)
w_model  = widgets.RadioButtons(options=['Random Forest', 'XGBoost', 'LightGBM', 'All Three'],
                                  value='All Three', description='Model:', style=style)
w_btn    = widgets.Button(description='▶  Predict', button_style='success',
                           layout=widgets.Layout(width='160px', height='36px'))
out      = widgets.Output()

def build_vec(feature_names):
    h, m, dow = w_hour.value, 0, w_dow.value
    vec = {f: 0.0 for f in feature_names}
    vec.update({
        'hour': h, 'minute': m, 'dow': dow,
        'hour_sin': np.sin(2*np.pi*h/24), 'hour_cos': np.cos(2*np.pi*h/24),
        'dow_sin':  np.sin(2*np.pi*dow/7),'dow_cos':  np.cos(2*np.pi*dow/7),
        'is_morning': int(5<=h<12), 'is_afternoon': int(12<=h<17),
        'is_evening': int(17<=h<22), 'is_night': int(h>=22 or h<5),
        'is_weekday': int(dow<5),
        'is_occupied': int(w_occ.value), 'people_count': w_people.value,
        'occ_confidence': 0.9 if w_occ.value else 0.1,
        'outdoor_temp': w_temp.value, 'outdoor_hum': w_hum.value,
        'is_raining': int(w_rain.value), 'is_sunny': int(w_sunny.value),
        'light_bedroom_on': int(w_light_br.value), 'light_living_room_on': int(w_light_lr.value),
        'light_kitchen_on': int(w_light_k.value),  'light_bathroom_on': int(w_light_ba.value),
        'climate_hvac_on': int(w_hvac.value),
        'climate_hvac_cur_temp': w_hcur.value, 'climate_hvac_tgt_temp': w_htgt.value,
        'hvac_temp_delta': w_htgt.value - w_hcur.value,
        'media_player_tv_on': int(w_tv.value), 'media_player_music_on': int(w_music.value),
        'fan_living_room_on': int(w_fan_lr.value), 'fan_bedroom_on': int(w_fan_br.value),
        'air_purifier_bedroom_on': int(w_purif.value),
        'lights_on_count': sum([int(w_light_br.value),int(w_light_lr.value),int(w_light_k.value),int(w_light_ba.value)]),
        'morning_routine': int(5<=h<9 and w_occ.value),
        'evening_routine': int(17<=h<22 and w_occ.value),
        'entertainment': int(w_tv.value or w_music.value),
        'sleep_mode': int(h>=22 or h<6),
    })
    return np.array([vec.get(f, 0.0) for f in feature_names], dtype=np.float32)

def on_predict(b):
    out.clear_output()
    with out:
        x       = build_vec(names)
        show_rf  = w_model.value in ('Random Forest', 'All Three')
        show_xgb = w_model.value in ('XGBoost', 'All Three')
        show_lgb = w_model.value in ('LightGBM', 'All Three')
        n_plots  = int(show_rf) + int(show_xgb) + int(show_lgb)
        fig, axes = plt.subplots(1, n_plots, figsize=(7*n_plots, 5), facecolor=BG)
        if n_plots == 1: axes = [axes]
        ax_idx = 0
        for model_name, model, le_use, color, do_show in [
            ('Random Forest', rf_fine,  le_fine, ACCENT,   w_model.value in ('Random Forest', 'All Three')),
            ('XGBoost',       xgb_fine, le_fine, ACCENT2,  w_model.value in ('XGBoost',       'All Three')),
            ('LightGBM',      lgb_fine, le_fine, '#c084fc', w_model.value in ('LightGBM',      'All Three')),
        ]:
            if not do_show: continue
            ax = axes[ax_idx]; ax_idx += 1
            ax.set_facecolor(CARD)
            proba   = pad_proba(model.predict_proba(x.reshape(1,-1)), model.classes_, n_cls)[0]
            top_idx = np.argsort(proba)[::-1][:8]
            lbls    = [le_use.classes_[i].replace('_',' ') for i in top_idx]
            vals    = proba[top_idx] * 100
            bars    = ax.barh(lbls[::-1], vals[::-1], color=color, zorder=3, alpha=0.85)
            for bar, v in zip(bars, vals[::-1]):
                ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                        f'{v:.1f}%', va='center', color=TEXT, fontsize=9)
            ax.set_xlim(0, 105); ax.set_xlabel('Confidence %', color=MUTED)
            ax.set_title(model_name, color=TEXT, fontsize=11)
            ax.tick_params(colors=MUTED, labelsize=8)
            ax.spines[:].set_color(BORDER)
            ax.grid(axis='x', color=BORDER, linestyle='--', alpha=0.4, zorder=0)
            bars[-1].set_edgecolor(color); bars[-1].set_linewidth(2)
        fig.suptitle(f'Hour={w_hour.value}:00  {["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][w_dow.value]}  Occupied={w_occ.value}',
                     color=WARN, fontsize=11)
        fig.tight_layout(rect=[0,0,1,0.94])
        plt.show()

w_btn.on_click(on_predict)

left_col  = widgets.VBox([w_hour, w_dow, w_occ, w_people, w_temp, w_hum, w_rain, w_sunny])
right_col = widgets.VBox([w_light_br, w_light_lr, w_light_k, w_light_ba,
                           w_hvac, w_hcur, w_htgt, w_tv, w_music, w_fan_lr, w_fan_br, w_purif])
controls  = widgets.HBox([left_col, right_col])
display(widgets.VBox([widgets.HTML('<h3 style="color:#22c9a4">🏠 Live Next-Action Predictor</h3>'),
                      controls, w_model, w_btn, out]))
on_predict(None)

In [ ]:
# ── Cell 8: Save models ───────────────────────────────────────────────────────
import pickle
models_to_save = {
    'rf_fine.pkl':     rf_fine,
    'xgb_fine.pkl':    xgb_fine,
    'lgb_fine.pkl':    lgb_fine,
    'rf_grouped.pkl':  rf_grp,
    'xgb_grouped.pkl': xgb_grp,
    'lgb_grouped.pkl': lgb_grp,
    'le_fine.pkl':     le_fine,
    'le_grouped.pkl':  le_grp,
}
for fname, obj in models_to_save.items():
    with open(fname, 'wb') as f: pickle.dump(obj, f)
    print(f'  Saved {fname} ({os.path.getsize(fname)/1e6:.1f} MB)')

import json
with open('model_metadata.json', 'w') as f:
    json.dump({'feature_names': names, 'fine_classes': le_fine.classes_.tolist(),
               'grouped_classes': le_grp.classes_.tolist(),
               'accuracy': {'RF_fine': round(rf_acc,4), 'XGB_fine': round(xg_acc,4), 'LGB_fine': round(lg_acc,4),
                            'RF_grouped': round(grf_acc,4), 'XGB_grouped': round(gxg_acc,4), 'LGB_grouped': round(glg_acc,4)}}, f, indent=2)

# Download all
from google.colab import files
for fname in list(models_to_save.keys()) + ['model_metadata.json']:
    files.download(fname)
print('✅ All models downloaded!')